# Bioresearch Drug Discovery Lab — GRPO Training (Colab)

End-to-end Colab notebook that trains a small instruct model with **GRPO (Group Relative Policy Optimization)** against the live Bioresearch OpenEnv server, using the long-horizon lab tasks (`protein_hypothesis_lab`, `target_discovery_lab`, `curriculum_self_play`).

**What this notebook does**

1. Installs Unsloth + TRL + the Bioresearch repo.
2. Starts the OpenEnv server in the background (inside the Colab VM).
3. Defines a **dense reward function** that rolls out an agent episode against the server and returns the terminal + per-step aggregated reward.
4. Loads `Qwen/Qwen2.5-1.5B-Instruct` with Unsloth 4-bit + LoRA.
5. Runs a short GRPO training loop (~150 steps ≈ 45 min on a free T4).
6. Plots the reward curve before vs. after training — the headline hackathon deliverable.

> The server is **deterministic** (tools return fixed data), so every `(prompt, response)` pair yields the same reward — exactly what GRPO needs.

## 1 · Setup

Install dependencies and clone the Bioresearch environment repo. The first cell assumes a fresh Colab runtime with a GPU attached.

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.26" "trl<0.11" peft accelerate bitsandbytes
!pip install -q httpx fastapi uvicorn pydantic pandas matplotlib

import os, subprocess, sys

# Point this at your fork of the environment before running.
# Either a HuggingFace Space URL or any git remote works — we clone, add to
# sys.path, and let Cell 4 boot the server from this working copy. Override
# by exporting BIORESEARCH_REPO_URL in the Colab environment.
REPO_URL = os.environ.get(
    "BIORESEARCH_REPO_URL",
    "https://huggingface.co/spaces/anirudhchida/bioresearch",
)

if not os.path.isdir("bioresearch"):
    if "anirudhchida" in REPO_URL:
        raise RuntimeError(
            "Set BIORESEARCH_REPO_URL (or edit REPO_URL above) to your fork of "
            "the bioresearch repo — the placeholder URL will 404 on clone."
        )
    subprocess.run(["git", "clone", REPO_URL, "bioresearch"], check=True)

sys.path.insert(0, "/content/bioresearch")
print("✅ Setup complete")

^C
ERROR: Operation cancelled by user
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [23 lines of output]
      Traceback (most recent call last):
        File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
          ~~~~^^
        File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 143, in get_requires_for_build_wheel
          return hook(config_settings)
        File "/private/var/folders

Cloning into 'bioresearch'...


⚠️  Replace YOUR_HF_USERNAME in the clone URL, or upload the repo manually to /content/bioresearch
✅ Setup complete


remote: Repository not found
fatal: repository 'https://huggingface.co/spaces/YOUR_HF_USERNAME/bioresearch/' not found


## 2 · Start the OpenEnv server

We launch `uvicorn` in a background subprocess so the reward function can hit `http://127.0.0.1:8000` during training.

In [ ]:
import subprocess, time, httpx

server_proc = subprocess.Popen(
    ["uvicorn", "server.app:app", "--host", "127.0.0.1", "--port", "8000"],
    cwd="/content/bioresearch",
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

# Wait for the server to accept connections.
for _ in range(40):
    try:
        r = httpx.get("http://127.0.0.1:8000/info", timeout=1.0)
        if r.status_code == 200:
            print("✅ Server is up:", r.json())
            break
    except Exception:
        time.sleep(1.0)
else:
    raise RuntimeError("OpenEnv server failed to start")

## 3 · Reward function — rollout a full lab episode

For every completion the policy produces we:

1. Parse it as a JSON tool-call or submission.
2. Replay up to `MAX_STEPS` steps against the OpenEnv server (same `task_id` every rollout — deterministic).
3. Return the **aggregated reward** (per-step process rewards + terminal reward).

This is the "dense signal" GRPO needs. A single-turn reward function would work too but collapses to the classifier we already have.

In [ ]:
import json, re
from client import BioresearchEnv
from models import BioresearchAction

SERVER_URL = "http://127.0.0.1:8000"
TASK_TYPE = "protein_hypothesis_lab"   # swap for target_discovery_lab / curriculum_self_play

env_client = BioresearchEnv(base_url=SERVER_URL)

# Non-greedy outer match first so a completion with trailing tokens after the
# JSON object doesn't accidentally swallow them into the payload. The greedy
# fallback rescues nested objects (e.g. intervention dicts inside the answer).
JSON_BLOCK_RE = re.compile(r"\{[\s\S]*?\}")
JSON_FULL_RE = re.compile(r"\{[\s\S]*\}")


def parse_completion(text: str):
    """Extract the first JSON object from the model output.

    On parse failure we return a minimal submit so the reward function still
    scores in `[0.01, 0.99]`. Critically, we do NOT inject the raw completion
    as the `reasoning` field — a missing `reasoning` key should correctly earn
    the zero-reasoning penalty in `grade_process_trace` rather than scoring
    artificially high via raw-token overlap.
    """
    text = text or ""
    for pattern in (JSON_BLOCK_RE, JSON_FULL_RE):
        m = pattern.search(text)
        if not m:
            continue
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError:
            continue
    return {"submit": True, "answer": ""}


def reward_lab_episode(prompts, completions, **kwargs) -> list[float]:
    """TRL GRPO reward function — one scalar reward per completion.

    Immediate-submit rollout: each completion is graded as a single agent
    turn. We deliberately do NOT loop `MAX_STEPS>1` replaying the same parsed
    action — after the first step the tool signature hits `lab.seen_calls`
    and `STEP_REWARD_TOOL_REDUNDANT` (-0.010/step) drains the reward. The
    per-step shaping curriculum lives in the multi-generation rollout variant.

    When the training dataset carries a `task_id` column (see Cell 8), TRL
    forwards it through `**kwargs` so we can `reset(task_id=...)` against the
    exact brief the model was prompted with. The environment is deterministic,
    so resetting on the same id always returns the same question — without
    this, completions get graded against a random *other* brief and the GRPO
    advantage signal collapses into noise.
    """
    task_ids = kwargs.get("task_id") or []
    task_types = kwargs.get("task_type") or []
    rewards: list[float] = []
    for idx, comp in enumerate(completions):
        text = comp if isinstance(comp, str) else comp[0]["content"]
        parsed = parse_completion(text)

        this_id = task_ids[idx] if idx < len(task_ids) else None
        this_type = task_types[idx] if idx < len(task_types) else TASK_TYPE
        if this_id:
            obs = env_client.reset(task_id=this_id, task_type=this_type)
        else:
            obs = env_client.reset(task_type=this_type)

        action = BioresearchAction(
            task_id=obs.task_id,
            submit=True,
            answer=parsed.get("answer", "") or "",
            reasoning=parsed.get("reasoning"),
            go_terms=parsed.get("go_terms"),
            proposed_intervention=parsed.get("intervention"),
        )
        result = env_client.step(action)
        rewards.append(float(result.reward or 0.0))
    return rewards


# Sanity-check on a trivial completion — no task_id kwarg, so the function
# falls back to a random reset. The point is just to confirm it returns a
# float in [0.01, 0.99] without crashing.
print("Sanity reward:", reward_lab_episode(["dummy"], ['{"submit": true, "answer": "unknown"}']))

## 4 · Build the training prompt dataset

Each training "prompt" is just an opening brief for `protein_hypothesis_lab`. The policy produces a JSON tool-call or submission per completion, and GRPO ranks them against each other.

In [ ]:
from datasets import Dataset

SYSTEM = (
    "You are a biomedical research agent. Respond with a SINGLE JSON object and nothing else.\n"
    "Call a tool with: {\"tool\": \"get_ppi\", \"args\": {\"gene\": \"TP53\"}}\n"
    "Or submit with: {\"submit\": true, \"answer\": \"<disease>\", \"reasoning\": \"...\", "
    "\"go_terms\": [\"GO:xxxxxxx\"], \"intervention\": {\"mode\": \"inhibit\", \"target\": \"TP53\"}}\n"
    "Tools: get_interpro, get_ppi, get_go, get_sequence, get_pathway, get_subcellular_location, search_catalogue."
)

# Pull 16 distinct opening briefs. We stash `task_id` + `task_type` alongside
# each prompt so the reward function can reset the env against the *exact*
# brief the model was shown (TRL forwards dataset columns through **kwargs).
prompts = []
for i in range(16):
    obs = env_client.reset(task_type=TASK_TYPE)
    prompts.append({
        "prompt": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": obs.question[:1800]},
        ],
        "task_id": obs.task_id,
        "task_type": TASK_TYPE,
    })

train_ds = Dataset.from_list(prompts)
print(train_ds)

## 5 · Load the model with Unsloth (4-bit + LoRA)

We use `Qwen2.5-1.5B-Instruct` because it fits on a free T4 and is small enough to show meaningful policy updates in ~45 minutes. Swap for `Qwen2.5-7B-Instruct` on an A100 for the deluxe demo.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
tokenizer.padding_side = "left"
print("✅ Model loaded")

## 6 · GRPO training loop

We use TRL's `GRPOTrainer` with our custom reward function. `num_generations=4` gives the algorithm a tight group of samples to rank per prompt.

In [ ]:
import torch
from trl import GRPOConfig, GRPOTrainer

# Colab T4 (Turing, CC 7.5) has no hardware bfloat16; fall back to fp16 there.
# A100/H100 and TPU-VM auto-upgrade to bf16 which has a wider dynamic range
# and avoids the fp16 loss-scaling dance.
_BF16_OK = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

grpo_config = GRPOConfig(
    output_dir="grpo_bioresearch",
    learning_rate=5e-6,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=256,
    max_steps=150,
    logging_steps=1,
    save_strategy="steps",
    save_steps=50,
    bf16=_BF16_OK,
    fp16=not _BF16_OK,
    beta=0.04,
    max_grad_norm=1.0,
    seed=42,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=grpo_config,
    train_dataset=train_ds,
    reward_funcs=[reward_lab_episode],
)

print("✅ Trainer ready")

In [ ]:
trainer.train()

## 7 · Plot the reward curve (the hackathon deliverable)

We pull the per-step reward trace from TRL's log history and plot a smoothed curve.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

log_df = pd.DataFrame(trainer.state.log_history)
reward_col = next((c for c in log_df.columns if "reward" in c and "std" not in c), None)
print("Available columns:", list(log_df.columns))
print("Using reward column:", reward_col)

if reward_col and "step" in log_df.columns:
    # TRL log_history interleaves rows with only eval/loss columns — drop them
    # so raw and EMA align with the actual training-step axis.
    subset = log_df[["step", reward_col]].dropna().reset_index(drop=True)
    steps = subset["step"]
    series = subset[reward_col]
    smooth = series.rolling(window=10, min_periods=1).mean()

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.plot(steps, series, alpha=0.35, label="reward (raw)")
    ax.plot(steps, smooth, color="#1f77b4", linewidth=2.5, label="reward (10-step EMA)")
    ax.set_xlabel("GRPO step")
    ax.set_ylabel("Mean episode reward")
    ax.set_title(f"Bioresearch Drug Discovery Lab — GRPO on {TASK_TYPE}")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("grpo_reward_curve.png", dpi=150)
    plt.show()

    baseline = float(series.head(10).mean())
    final = float(series.tail(10).mean())
    print(f"Baseline (first 10 steps): {baseline:.4f}")
    print(f"Final    (last 10 steps):  {final:.4f}")
    print(f"Delta                    : {final - baseline:+.4f}")
else:
    print("⚠️  No reward column found — check TRL version/logging setup.")

## 7b · Second reward curve — `perturbation_qa`

`perturbation_qa` is the crispest GRPO signal in the suite: each episode is a batch of ~10 binary CRISPRi pairs, the reward is continuous macro-F1 + balanced accuracy (`grade_perturbation_batch`), prompts are tiny, and answers are single tokens. This cell trains the same model for a short burst against this task so we can overlay a second curve next to `protein_hypothesis_lab`.


In [ ]:
from datasets import Dataset

PERT_SYSTEM = (
    "You are a biomedical world model. Given a batch of CRISPRi perturbation pairs, "
    "respond with a SINGLE JSON object of the form "
    '{"perturbation_answers": {"<pair_id>": true/false, ...}} '
    "where true means knocking down query_gene changes target_gene in the given cell line."
)


def _pert_reward(prompts, completions, **kwargs):
    """Grade each completion against the *same* batch its prompt came from.

    TRL forwards dataset columns through `**kwargs`, so we pull `task_id` out
    and reset the deterministic env on that id — otherwise the completion
    gets graded against a fresh random batch and the GRPO advantage is noise.
    """
    task_ids = kwargs.get("task_id") or []
    rewards = []
    for idx, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        match = JSON_FULL_RE.search(text or "")
        answers: dict[str, bool] = {}
        if match:
            try:
                payload = json.loads(match.group(0))
                raw = payload.get("perturbation_answers") or payload.get("answers") or {}
                if isinstance(raw, dict):
                    for k, v in raw.items():
                        if isinstance(v, bool):
                            answers[str(k)] = v
                        elif isinstance(v, str):
                            answers[str(k)] = v.strip().lower() in ("yes", "true", "1")
            except Exception:
                answers = {}

        this_id = task_ids[idx] if idx < len(task_ids) else None
        if this_id:
            obs = env_client.reset(task_id=this_id, task_type="perturbation_qa")
        else:
            obs = env_client.reset(task_type="perturbation_qa")
        result = env_client.step(BioresearchAction(
            task_id=obs.task_id,
            perturbation_answers=answers,
        ))
        rewards.append(float(result.reward or 0.0))
    return rewards


pert_prompts = []
for _ in range(32):
    obs = env_client.reset(task_type="perturbation_qa")
    batch_lines = [
        f"- {p['pair_id']}: does knocking down {p['query_gene']} change {p['target_gene']} in {p['cell_line']}?"
        for p in (obs.perturbation_batch or [])
    ]
    user = "Answer yes/no for each pair:\n" + "\n".join(batch_lines)
    pert_prompts.append({
        "prompt": [
            {"role": "system", "content": PERT_SYSTEM},
            {"role": "user", "content": user},
        ],
        "task_id": obs.task_id,
        "task_type": "perturbation_qa",
    })

pert_dataset = Dataset.from_list(pert_prompts)

pert_config = GRPOConfig(
    output_dir="grpo_perturbation_qa",
    learning_rate=5e-6,      # guide warns 1e-5 can diverge on 1.5B within 50 steps
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=256,
    max_steps=60,            # 32-row dataset @ accum=2 only gives 16 epoch steps — bump to 60
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    bf16=_BF16_OK,
    fp16=not _BF16_OK,
    beta=0.04,
    max_grad_norm=1.0,
    seed=42,
)

pert_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=pert_config,
    train_dataset=pert_dataset,
    reward_funcs=[_pert_reward],
)
pert_trainer.train()

pert_df = pd.DataFrame(pert_trainer.state.log_history)
pert_reward_col = next(
    (c for c in pert_df.columns if "reward" in c.lower() and "std" not in c.lower()),
    None,
)
if pert_reward_col and "step" in pert_df.columns:
    pert_subset = pert_df[["step", pert_reward_col]].dropna().reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    if reward_col and reward_col in log_df.columns:
        lab_subset = log_df[["step", reward_col]].dropna().reset_index(drop=True)
        ax.plot(lab_subset["step"], lab_subset[reward_col], label="protein_hypothesis_lab", alpha=0.75)
    ax.plot(pert_subset["step"], pert_subset[pert_reward_col], label="perturbation_qa", alpha=0.9)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Mean reward")
    ax.set_title("GRPO reward curves — lab vs. perturbation_qa")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("grpo_reward_curves.png", dpi=120)
    plt.show()

    s = pert_subset[pert_reward_col]
    pert_baseline = float(s.head(10).mean())
    pert_final = float(s.tail(10).mean())
    print(f"perturbation_qa baseline (first 10): {pert_baseline:.4f}")
    print(f"perturbation_qa final    (last 10):  {pert_final:.4f}")
    print(f"perturbation_qa delta               : {pert_final - pert_baseline:+.4f}")


## 7c · Third reward curve — `perturbation_direction_qa` (v3, 3-class)

`perturbation_direction_qa` is the v3 upgrade of the binary perturbation task: instead of yes/no, the agent must predict `Increase` / `Decrease` / `Unknown` for every CRISPRi pair. Three labels means meaningfully more reward entropy per step, so the GRPO advantage estimate is larger and the reward curve should climb faster than the binary task. This cell trains a short burst and overlays the third curve on the chart — the final hackathon deliverable for "Reward Improvement".

In [ ]:
from datasets import Dataset

DIR_SYSTEM = (
    "You are a CRISPRi directional world model. Given a batch of (query_gene, target_gene, cell_line) "
    "pairs, respond with a SINGLE JSON object of the form "
    '{"direction_answers": {"<pair_id>": "Increase"|"Decrease"|"Unknown", ...}} '
    "covering every pair_id in the batch."
)


def _dir_reward(prompts, completions, **kwargs):
    """Grade each completion against the exact directional batch it saw.

    Like `_pert_reward`, we read `task_id` from the dataset column that TRL
    forwards via `**kwargs`; resetting the env on the same id replays the
    identical batch (the environment is deterministic).
    """
    task_ids = kwargs.get("task_id") or []
    rewards = []
    for idx, completion in enumerate(completions):
        text = completion[0]["content"] if isinstance(completion, list) else completion
        match = JSON_FULL_RE.search(text or "")
        answers: dict[str, str] = {}
        if match:
            try:
                payload = json.loads(match.group(0))
                raw = payload.get("direction_answers") or payload.get("answers") or {}
                if isinstance(raw, dict):
                    for k, v in raw.items():
                        if isinstance(v, str):
                            answers[str(k)] = v.strip()
            except Exception:
                answers = {}

        this_id = task_ids[idx] if idx < len(task_ids) else None
        if this_id:
            obs = env_client.reset(task_id=this_id, task_type="perturbation_direction_qa")
        else:
            obs = env_client.reset(task_type="perturbation_direction_qa")
        result = env_client.step(BioresearchAction(
            task_id=obs.task_id,
            direction_answers=answers,
        ))
        rewards.append(float(result.reward or 0.0))
    return rewards


dir_prompts = []
for _ in range(32):
    obs = env_client.reset(task_type="perturbation_direction_qa")
    batch_lines = [
        f"- {p['pair_id']}: knocking down {p['query_gene']} -> target {p['target_gene']} in {p['cell_line']}?"
        for p in (obs.direction_batch or [])
    ]
    user = (
        "Predict 'Increase', 'Decrease', or 'Unknown' for each pair:\n"
        + "\n".join(batch_lines)
    )
    dir_prompts.append({
        "prompt": [
            {"role": "system", "content": DIR_SYSTEM},
            {"role": "user", "content": user},
        ],
        "task_id": obs.task_id,
        "task_type": "perturbation_direction_qa",
    })

dir_dataset = Dataset.from_list(dir_prompts)

dir_config = GRPOConfig(
    output_dir="grpo_perturbation_direction_qa",
    learning_rate=5e-6,      # match the guide; 1e-5 can diverge within 50 steps on 1.5B
    per_device_train_batch_size=1,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_prompt_length=1024,
    max_completion_length=256,
    max_steps=60,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    bf16=_BF16_OK,
    fp16=not _BF16_OK,
    beta=0.04,
    max_grad_norm=1.0,
    seed=42,
)

dir_trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=dir_config,
    train_dataset=dir_dataset,
    reward_funcs=[_dir_reward],
)
dir_trainer.train()

dir_df = pd.DataFrame(dir_trainer.state.log_history)
dir_reward_col = next(
    (c for c in dir_df.columns if "reward" in c.lower() and "std" not in c.lower()),
    None,
)

fig, ax = plt.subplots(figsize=(9, 4.5))
if reward_col and reward_col in log_df.columns:
    lab_subset = log_df[["step", reward_col]].dropna().reset_index(drop=True)
    ax.plot(lab_subset["step"], lab_subset[reward_col], label="protein_hypothesis_lab", alpha=0.75)
if pert_reward_col and pert_reward_col in pert_df.columns:
    pert_subset2 = pert_df[["step", pert_reward_col]].dropna().reset_index(drop=True)
    ax.plot(pert_subset2["step"], pert_subset2[pert_reward_col], label="perturbation_qa (binary)", alpha=0.85)
if dir_reward_col and dir_reward_col in dir_df.columns:
    dir_subset = dir_df[["step", dir_reward_col]].dropna().reset_index(drop=True)
    ax.plot(
        dir_subset["step"],
        dir_subset[dir_reward_col],
        label="perturbation_direction_qa (3-class, v3)",
        alpha=0.95,
        linewidth=2.0,
    )
ax.set_xlabel("Training step")
ax.set_ylabel("Mean reward")
ax.set_title("GRPO reward curves — lab vs binary vs 3-class directional")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("grpo_reward_curves_v3.png", dpi=120)
plt.show()

if dir_reward_col and "step" in dir_df.columns:
    s = dir_subset[dir_reward_col]
    dir_baseline = float(s.head(10).mean())
    dir_final = float(s.tail(10).mean())
    print(f"perturbation_direction_qa baseline (first 10): {dir_baseline:.4f}")
    print(f"perturbation_direction_qa final    (last 10):  {dir_final:.4f}")
    print(f"perturbation_direction_qa delta               : {dir_final - dir_baseline:+.4f}")

## 7d · Held-out evaluation

Training-time reward is a noisy signal. Before saving the adapter we roll it out on 5 fresh briefs per task (deterministic env, but different `task_id`s from the ones TRL sampled during training) and report `mean ± std`. This (a) quantifies the lift the curves suggest, (b) catches overfitting on the 16–32 row training sets, and (c) fills the `Reward Improvement` cells in the README baseline table with concrete numbers that get written to `eval_summary.json`.

In [ ]:
import numpy as np
import torch

EVAL_N = 5  # held-out rollouts per task


def _collect_eval_ids(task_type: str, n: int = EVAL_N, cap: int = 24) -> list[str]:
    """Sample up to `n` distinct deterministic task_ids for the given task."""
    seen: list[str] = []
    for _ in range(cap):
        obs = env_client.reset(task_type=task_type)
        if obs.task_id not in seen:
            seen.append(obs.task_id)
        if len(seen) >= n:
            break
    return seen


def _model_completion(messages, max_new_tokens: int = 256) -> str:
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)


eval_summary: dict[str, dict] = {}

# Curve 1 — protein_hypothesis_lab
lab_ids = _collect_eval_ids("protein_hypothesis_lab")
lab_rewards: list[float] = []
for tid in lab_ids:
    obs = env_client.reset(task_id=tid, task_type="protein_hypothesis_lab")
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": obs.question[:1800]},
    ]
    completion = _model_completion(msgs)
    r = reward_lab_episode(
        prompts=[msgs], completions=[completion],
        task_id=[tid], task_type=["protein_hypothesis_lab"],
    )
    lab_rewards.extend(r)
eval_summary["protein_hypothesis_lab"] = {
    "mean": float(np.mean(lab_rewards)), "std": float(np.std(lab_rewards)),
    "n": len(lab_rewards), "task_ids": lab_ids,
}

# Curve 2 — perturbation_qa
pert_ids = _collect_eval_ids("perturbation_qa")
pert_rewards: list[float] = []
for tid in pert_ids:
    obs = env_client.reset(task_id=tid, task_type="perturbation_qa")
    batch_lines = [
        f"- {p['pair_id']}: does knocking down {p['query_gene']} change {p['target_gene']} in {p['cell_line']}?"
        for p in (obs.perturbation_batch or [])
    ]
    msgs = [
        {"role": "system", "content": PERT_SYSTEM},
        {"role": "user", "content": "Answer yes/no for each pair:\n" + "\n".join(batch_lines)},
    ]
    completion = _model_completion(msgs)
    r = _pert_reward(
        prompts=[msgs], completions=[completion],
        task_id=[tid], task_type=["perturbation_qa"],
    )
    pert_rewards.extend(r)
eval_summary["perturbation_qa"] = {
    "mean": float(np.mean(pert_rewards)), "std": float(np.std(pert_rewards)),
    "n": len(pert_rewards), "task_ids": pert_ids,
}

# Curve 3 — perturbation_direction_qa
dir_ids = _collect_eval_ids("perturbation_direction_qa")
dir_rewards_eval: list[float] = []
for tid in dir_ids:
    obs = env_client.reset(task_id=tid, task_type="perturbation_direction_qa")
    batch_lines = [
        f"- {p['pair_id']}: knocking down {p['query_gene']} -> target {p['target_gene']} in {p['cell_line']}?"
        for p in (obs.direction_batch or [])
    ]
    msgs = [
        {"role": "system", "content": DIR_SYSTEM},
        {"role": "user", "content": "Predict 'Increase', 'Decrease', or 'Unknown' for each pair:\n" + "\n".join(batch_lines)},
    ]
    completion = _model_completion(msgs)
    r = _dir_reward(
        prompts=[msgs], completions=[completion],
        task_id=[tid], task_type=["perturbation_direction_qa"],
    )
    dir_rewards_eval.extend(r)
eval_summary["perturbation_direction_qa"] = {
    "mean": float(np.mean(dir_rewards_eval)), "std": float(np.std(dir_rewards_eval)),
    "n": len(dir_rewards_eval), "task_ids": dir_ids,
}

print(f"{'task':30s}  {'mean':>8s}  {'std':>8s}  {'n':>3s}")
print("-" * 55)
for name, stats in eval_summary.items():
    print(f"{name:30s}  {stats['mean']:>8.4f}  {stats['std']:>8.4f}  {stats['n']:>3d}")

with open("eval_summary.json", "w") as f:
    json.dump(eval_summary, f, indent=2)
print("\n✅ Saved eval_summary.json")

## 8 · Save + teardown

Save the LoRA adapter and shut the server down cleanly.

In [ ]:
model.save_pretrained("grpo_bioresearch_lora")
tokenizer.save_pretrained("grpo_bioresearch_lora")

server_proc.terminate()
server_proc.wait(timeout=5)
print("✅ LoRA saved + server stopped")